In [2]:
%autoreload 2

In [1]:
%reload_ext autoreload
import os, sys, random
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib import  rcParams
from fish import Gafftopsail
sys.path.append(r'C://Users//Zichen//anaconda3//envs//2ptank//Lib//site-packages//')
import utils, barcode
path = (r'C:/Data/Imaging\260425_overlap//fish5//')
rcParams['font.size'] = 12

In [ ]:
fish = Gafftopsail(path, filelist = ['stimulus', 'imaging', 'alignment'], sequence = 5)
fish.stimulus_df.stim_name = [str(s) for s in fish.stimulus_df.stim_name]

__EACH STIM__

In [ ]:
#axis 0: trial; axis 1: neuron; axis 2: frame
#from stationary start to duration + 20
f_pertrial_dict = barcode.get_pertrial_f(fish)

__Calculate the DSI of the neuron__

In [ ]:
dot_stim = None
stim_responses = barcode.get_grating_dsi(fish,f_pertrial_dict,baseline_s=7,response_s=15, dot_stim = dot_stim)

__barcoding 2: large response__

In [ ]:
barred_gratingneurons = barcode.select_gratingbarcode(fish, f_pertrial_dict, baseline_s = 7, response_s =15, perc_trial_threshold = 1)

In [ ]:
#barred_gratingneurons = barcode.sort_gratingneurons(fish, f_pertrial_dict, barred_gratingneurons)

__rainbow plot__

In [ ]:
#combine barcoding and stim_responses
for stim, neurons in barred_gratingneurons.items():
    stim_responses[f'barred_{stim}'] = stim_responses.index.isin(neurons)

#save it
os.makedirs(os.path.join(fish.path, "analyze_results"), exist_ok=True)
if dot_stim is not None:
    stim_responses.to_csv(fish.path + f'//analyze_results//{dot_stim}grating_responses.csv')
else:
    stim_responses.to_csv(fish.path + f'//analyze_results//grating_responses.csv')

In [ ]:
# barcode.plot_rainbow(fish, stim_responses)
# plt.savefig(fish.path + f'//Graphs//grating_rainbow.png')
# plt.show()
# plt.close()

In [ ]:
fish.stimulus_df.loc[:, "somegratingresponse"] = fish.stimulus_df.okr_eye | fish.stimulus_df.omr_tail

__Look at response during overlap__

In [2]:
def get_f_efference(fish, highDSI, barred_gratingneurons):
    """Get the avg trace of a population of cells"""
    #look at vis responsive cells
    region_list = [['prosencephalon_(forebrain)', 'mesencephalon_(midbrain)'], 'rhombencephalon_(hindbrain)', 'tectum', 'pretectum']

    #get traces
    f_dict = {'forebrain':{}, 'hindbrain':{}, 'tectum': {}, 'pretectum': {}}

    fig, ax = plt.subplots(len(barred_gratingneurons), 1 + 2 * len(region_list), figsize = (20, 20))
    #each direction: line vs dot line
    for s, stim in enumerate(barred_gratingneurons):
        stim_n = set(barred_gratingneurons[stim])
        #DSI selection
        if highDSI:
            DSI_boundary = 0.5
            angle_boundary = 45
        else:
            DSI_boundary = 0
            angle_boundary = 1000
        dsi_n = set(stim_responses[(stim_responses['DSI2'] > DSI_boundary) &
                                   (np.abs((stim_responses['dir'] - utils.omr_angles[stim] + 180) % 360 - 180) <= angle_boundary)].index)
        print(f"DSI > {DSI_boundary}, angle within +/-{angle_boundary}deg")
        ax_imshow = ax[s, 0]
        ax_imshow.imshow(fish.img_dict[2], origin = 'lower')
        ax_imshow.set_axis_off()
        for r, (region, region_c) in enumerate(zip(region_list, ['white', 'grey', 'yellow','green'])):
            ax_line = ax[s, r * 2 + 1]
            ax_heat = ax[s, r * 2 + 2]
            f_df = []
            if type(region) == list:
                region_n = set(fish.apos_all.index[fish.apos_all['regions'].apply(lambda x: any(ri in x for ri in region))])#set(fish.apos_all.index)#
            else:
                region_n = set(fish.apos_all.index[fish.apos_all['regions'].apply(lambda x: region in x)])
            region_n = list(region_n & stim_n & dsi_n)

            opp_stim = utils.angles_omr[(utils.omr_angles[stim] + 180)%360]
            ax_imshow.scatter(fish.pos_all.loc[region_n, 'xpos'], fish.pos_all.loc[region_n, 'ypos'], marker = ',', s = 1, color = region_c)
            for g, (grating, color) in enumerate(zip([stim, opp_stim], [utils.omr_colors[stim], utils.omr_colors[opp_stim]])):
                for d, (dot, linestyle) in enumerate(zip([None, 'dot_l', 'dot_r'], ['-', ':', ':'])):
                    if dot == None:
                        real_stim = grating
                    else:
                        real_stim =  str([dot, grating])
                    #find trials that have motor movements
                    trial_df = fish.stimulus_df[fish.stimulus_df.stim_name == real_stim].reset_index(drop = True)
                    motortrials = list(trial_df[trial_df.somegratingresponse].index)
                    nonmotortrials = list(trial_df[~trial_df.somegratingresponse].index)
                    for m, (triallist, motor) in enumerate(zip([motortrials, nonmotortrials], [True, False])):
                        #get fluorscnece
                        f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
                        f_mean = f_acrosstrial.mean(axis = 0)
                        f_sem = f_acrosstrial.std(axis=0) / np.sqrt(f_acrosstrial.shape[0])
                        df = pd.DataFrame(f_acrosstrial)
                        if len(df) > 0:
                            df.loc[:, 'barred'] = stim
                            df.loc[:, 'stim'] = real_stim
                            df.loc[:, 'gratingresponse'] = motor
                        f_df.append(df)
                        #start plotting
                        if not motor: actualcolor = 'grey'
                        else: actualcolor = color
                        ax_line.plot(f_mean, c=actualcolor, linestyle = linestyle)#color: grating, linestyle: dot, alpha: motor
                        ax_line.fill_between(np.arange(len(f_mean)),f_mean - f_sem, f_mean + f_sem,color=actualcolor,alpha=0.1, linewidth = 0)
                        ax_line.set_ylim([0.1, 0.6])
                        ax_heat.imshow(f_acrosstrial, aspect='auto',extent=[0, len(f_mean), (g * 3 + d) * 2 + m, (g * 3 + d) * 2 + m + 1],cmap='viridis', vmin = 0.2, vmax = 0.6)
                        ax_heat.axhline((g * 3 + d) * 2 + m, linestyle='--', color='white')
            ax_heat.set_ylim([0, (g * 3 + d) * 2 + m + 1])
            if s == 0:
                f_dict[list(f_dict.keys())[r]] = pd.concat(f_df, axis = 0, ignore_index = True)
            else:
                f_dict[list(f_dict.keys())[r]] = pd.concat([f_dict[list(f_dict.keys())[r]], pd.concat(f_df, axis = 0, ignore_index = True)], axis = 0, ignore_index = True)
    return f_dict

In [3]:
highDSI = True
f_dict = get_f_efference(fish, highDSI)
plt.show()
plt.close()

NameError: name 'fish' is not defined

In [4]:
#combine across fish
paths =[
 r"C:\Data\Imaging\260415_overlap\fish3",r"C:\Data\Imaging\260415_overlap\fish3_2",
r"C:\Data\Imaging\260415_overlap\fish4", r"C:\Data\Imaging\260415_overlap\fish4_2",
     r"C:\Data\Imaging\260415_overlap\fish6", r"C:\Data\Imaging\260415_overlap\fish6_2",
r"C:\Data\Imaging\260425_overlap\fish1", r"C:\Data\Imaging\260425_overlap\fish1_2",
r"C:\Data\Imaging\260425_overlap\fish2",r"C:\Data\Imaging\260425_overlap\fish2_2",
r"C:\Data\Imaging\260425_overlap\fish3",r"C:\Data\Imaging\260425_overlap\fish3_2",
r"C:\Data\Imaging\260425_overlap\fish5",r"C:\Data\Imaging\260425_overlap\fish5_2",
 r"C:\Data\Imaging\260425_overlap\fish6", r"C:\Data\Imaging\260425_overlap\fish6_2",
r"C:\Data\Imaging\260425_overlap\fish7",r"C:\Data\Imaging\260425_overlap\fish7_2",
r"C:\Data\Imaging\260425_overlap\fish8",r"C:\Data\Imaging\260425_overlap\fish8_2",
r"C:\Data\Imaging\260425_overlap\fish9", r"C:\Data\Imaging\260425_overlap\fish9_2",
 r"C:\Data\Imaging\260425_overlap\fish10", r"C:\Data\Imaging\260425_overlap\fish10_2",
r"C:\Data\Imaging\260425_overlap\fish11", r"C:\Data\Imaging\260425_overlap\fish11_2",]
for path in paths:
    fish = Gafftopsail(path + '//', filelist=['stimulus', 'imaging', 'alignment'], sequence=5)
    fish.stimulus_df.stim_name = [str(s) for s in fish.stimulus_df.stim_name]
    fish.stimulus_df.loc[:, "somegratingresponse"] = fish.stimulus_df.hunting_eye | fish.stimulus_df.hunting_tail

    #axis 0: trial; axis 1: neuron; axis 2: frame
    #from stationary start to duration + 20
    f_pertrial_dict = barcode.get_pertrial_f(fish)
    for dot_stim in [None, 'dot_l', 'dot_r']:
        stim_responses = barcode.get_grating_dsi(fish, f_pertrial_dict, baseline_s=7, response_s=12, dot_stim=dot_stim)
        if dot_stim == None:
            barred_gratingneurons = barcode.select_gratingbarcode(fish, f_pertrial_dict, baseline_s=7, response_s=12, perc_trial_threshold=1)
        # barred_gratingneurons = barcode.sort_gratingneurons(fish, f_pertrial_dict, barred_gratingneurons)

        #combine barcoding and stim_responses
        for stim, neurons in barred_gratingneurons.items():
            stim_responses[f'barred_{stim}'] = stim_responses.index.isin(neurons)
        #
        # #save it
        # os.makedirs(os.path.join(fish.path, "analyze_results"), exist_ok=True)
        # if dot_stim is not None:
        #     stim_responses.to_csv(fish.path + f'//analyze_results//{dot_stim}grating_responses_delayed.csv')
        # else:
        #     stim_responses.to_csv(fish.path + f'//analyze_results//grating_responses_delayed.csv')

        highDSI = True
        f_dict = get_f_efference(fish, highDSI, barred_gratingneurons)
        for region in ['forebrain', 'hindbrain', 'tectum', 'pretectum']:
                df = f_dict[region]
                df.to_csv(fish.path + f"//analyze_results//gratingefferenceh_{region}neurons_highDSI.csv")

        highDSI = False
        f_dict = get_f_efference(fish, highDSI, barred_gratingneurons)
        for region in ['forebrain', 'hindbrain', 'tectum', 'pretectum']:
                df = f_dict[region]
                df.to_csv(fish.path + f"//analyze_results//gratingefferenceh_{region}neurons.csv")

        plt.close()

low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
selection criteria: max >= baseline mean * 2
selection cri

C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\L

DSI > 0.5, angle within +/-45deg


C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\L

DSI > 0.5, angle within +/-45deg


...

C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)


DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg


C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)


DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg


C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)
C:\Users\Zichen\AppData\Local\Temp\ipykernel_8512\2194022928.py:50: RuntimeWarning: Mean of empty slice
  f_acrosstrial = np.nanmean(f_pertrial_dict[real_stim][triallist][:, region_n], axis = 0)


In [15]:
fish.stimulus_df

,stim_name,angle,velocity,stim_type,stationary_time,duration,texture,circle_center,circle_radius,angular_velocity,real_starttime,real_starttime_s,hunting_eye,hunting_tail,okr_eye,omr_tail,some_eye,some_tail,somegratingresponse
0,right,90,0.042,"[m, m]",20,35,"[rgb_circle, grating_rgb]","[[0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 17:37:07.704805,0.000000,True,True,True,True,True,True,True
1,forward,0,0.042,"[m, m]",20,35,"[rgb_circle, grating_rgb]","[[0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 17:37:42.749868,35.045063,True,True,False,True,True,True,True
2,left,270,0.042,"[m, m]",20,35,"[rgb_circle, grating_rgb]","[[0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 17:38:17.802183,70.097378,False,False,True,True,True,True,True
3,"['dot_r', 'left']","[-90, 270]","[0.05321356406289263, 0.042]","[m, m]","[25, 20]",35,"[rgb_circle, grating_rgb]","[[-279.9038105676658, -75], [0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 17:38:52.840536,105.135731,False,False,True,True,True,True,True
4,left,270,0.042,"[m, m]",20,35,"[rgb_circle, grating_rgb]","[[0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 17:39:27.881881,140.177076,True,True,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,"['dot_l', 'right']","[90, 90]","[0.05321356406289263, 0.042]","[m, m]","[25, 20]",35,"[rgb_circle, grating_rgb]","[[-279.9038105676658, 75], [0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 18:40:01.485804,3773.780999,True,True,False,False,True,True,False
101,"['dot_l', 'left']","[90, 270]","[0.05321356406289263, 0.042]","[m, m]","[25, 20]",35,"[rgb_circle, grating_rgb]","[[-279.9038105676658, 75], [0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 18:40:36.531138,3808.826333,False,False,False,True,True,True,True
102,"['dot_l', 'backward']","[90, 180]","[0.05321356406289263, 0.042]","[m, m]","[25, 20]",35,"[rgb_circle, grating_rgb]","[[-279.9038105676658, 75], [0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 18:41:11.560516,3843.855711,False,False,False,True,True,True,True
103,backward,180,0.042,"[m, m]",20,35,"[rgb_circle, grating_rgb]","[[0, 0]]","[3.2808248822221504, 3]","[0, 0]",2026-05-15 18:41:46.594878,3878.890073,False,False,False,False,True,False,False
